# Anatomy of a production prompt: role, context, task, format

**The Agent Build Log · Build #009 · Prompt Engineering 1/8**

Every prompt guide lists the same parts: a **role**, **context**, the **task** and an output **format**.
This build measures what each part is actually worth on one real job: triaging support tickets into
category, priority and sentiment.

1. [The data: a dev set and a held-out test set](#1-the-data-a-dev-set-and-a-held-out-test-set)
2. [Six prompts, one part at a time](#2-six-prompts-one-part-at-a-time)
3. [Run them](#3-run-them)
4. [One ticket, six prompts](#4-one-ticket-six-prompts)
5. [What each part is worth](#5-what-each-part-is-worth)
6. [Findings](#6-findings)

**Model:** Claude Haiku 5.5 with thinking off (classification doesn't need it; see build #008).
**Needs:** an Anthropic API key in a `.env` file. A full run costs about **4 cents**.

In [1]:
import json, os, re, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
import anthropic

load_dotenv(Path.cwd() / ".env") or load_dotenv(Path.cwd().parents[1] / ".env")
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in a .env file (see README)"
pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 90)

MODEL = "claude-haiku-5-5"
PRICE = {"input": 0.10, "output": 0.50}   # $ per 1M tokens (prompts up to 100K tokens)
client = anthropic.Anthropic().with_options(max_retries=6)
DATA = json.loads(Path("data/tickets.json").read_text())
FIELDS = DATA["fields"]
print(f"model: {MODEL} | dev {len(DATA['dev'])} tickets | test {len(DATA['test'])} tickets")

model: claude-haiku-5-5 | dev 20 tickets | test 12 tickets


## 1. The data: a dev set and a held-out test set

- **Dev (20 tickets):** the tickets and labels from build #004. I wrote the rubric below by looking at how I had labelled these.
- **Test (12 tickets):** new tickets, labelled **with the rubric, before any model saw them**. If the context only helps
  on dev, the rubric was overfitted to those 20 tickets. If it helps on test too, it's carrying real information.

The labels are mine, so the score is **agreement with my labelling rules**, not absolute truth. That's the point of
context in production: the model can't know your team's definitions unless you write them down.

In [2]:
pd.DataFrame([{"set": s, "id": t["id"], **t["label"], "ticket": t["text"][:70]} for s in ("dev", "test") for t in DATA[s][:3]])

,set,id,category,priority,sentiment,ticket
0,dev,1,billing,high,neutral,"Hi, I was charged twice for order ORD-48213. Please refund the extra $"
1,dev,2,bug,urgent,negative,YOUR CHECKOUT PAGE IS DOWN. Nobody on our team can pay. This is costin
2,dev,3,feature_request,low,positive,Would love a dark mode for the dashboard someday. Keep up the great wo
3,test,101,bug,urgent,negative,Our whole team gets a 500 error on the reports page since this morning
4,test,102,billing,high,neutral,"I got charged twice for ORD-81234, $64.00 each time. Please fix."
5,test,103,billing,low,neutral,Is there a student discount?


## 2. Six prompts, one part at a time

Four prompts build up one part at a time (**A → B → C → D**). Two more start from the full prompt **D** and remove
one part, to check that part still matters when everything else is there.

In [3]:
TASK = "Triage this support ticket: give its category, priority and sentiment."

FORMAT = ("Reply with only a JSON object, no other text:\n"
          '{"category": one of ' + json.dumps(FIELDS["category"]) + ', "priority": one of ' + json.dumps(FIELDS["priority"])
          + ', "sentiment": one of ' + json.dumps(FIELDS["sentiment"]) + "}")

CONTEXT = '''How our support team labels tickets:
Category
- billing: charges, refunds, plans, pricing questions, cancellations
- bug: something in the product is broken or wrong
- account: logging in, security, account settings, users
- shipping: delivery, tracking, damaged or wrong items
- feature_request: ideas for the product, and general feedback or praise
Priority
- urgent: many users blocked or money being lost right now, or a security incident (someone else in the account)
- high: one customer blocked with no workaround (locked out, charged twice for the same order, item lost or damaged,
  a crash on a main screen), or a business deadline at risk
- medium: a real problem with a workaround or a simple fix (wrong amount, coupon, cancellation, late tracking,
  exchange, a non-blocking bug), or a feature they would use right away
- low: questions, "someday" ideas, praise
Sentiment
- negative: frustration, or reports something broken, lost, damaged or blocked for them
- neutral: a calm request or question, including calm billing requests
- positive: praise or thanks is the main point'''

ROLE = "You are the support triage lead at an online electronics shop. You sort incoming tickets so the right team picks them up first."

def build(parts: set[str], ticket: str):
    user = [CONTEXT] if "context" in parts else []
    user += [TASK, f"<ticket>\n{ticket}\n</ticket>"]
    if "format" in parts:
        user.append(FORMAT)
    return (ROLE if "role" in parts else None), "\n\n".join(user)

VARIANTS = {
    "A: task only":              {"task"},
    "B: + format":               {"task", "format"},
    "C: + context":              {"task", "format", "context"},
    "D: + role (full prompt)":   {"task", "format", "context", "role"},
    "D without context":         {"task", "format", "role"},
    "D without format":          {"task", "context", "role"},
}
system, user = build(VARIANTS["D: + role (full prompt)"], DATA["dev"][0]["text"])
print("SYSTEM:", system, "\n\nUSER:\n" + user)

SYSTEM: You are the support triage lead at an online electronics shop. You sort incoming tickets so the right team picks them up first. 

USER:
How our support team labels tickets:
Category
- billing: charges, refunds, plans, pricing questions, cancellations
- bug: something in the product is broken or wrong
- account: logging in, security, account settings, users
- shipping: delivery, tracking, damaged or wrong items
- feature_request: ideas for the product, and general feedback or praise
Priority
- urgent: many users blocked or money being lost right now, or a security incident (someone else in the account)
- high: one customer blocked with no workaround (locked out, charged twice for the same order, item lost or damaged,
  a crash on a main screen), or a business deadline at risk
- medium: a real problem with a workaround or a simple fix (wrong amount, coupon, cancellation, late tracking,
  exchange, a non-blocking bug), or a feature they would use right away
- low: questions, "some

**Parsing.** A downstream system needs exactly one allowed value per field.
- If the reply has a JSON object (after stripping code fences), we read the three fields from it.
- Otherwise we look for allowed values as whole words. The reply is **usable** only if each field has exactly one.

That's generous to the free-text prompts: a human reading them might do better, a program won't.

In [4]:
def parse(text: str):
    body = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    m = re.search(r"\{.*\}", body, re.S)
    if m:
        try:
            d = json.loads(m.group(0))
            out = {f: str(d.get(f, "")).strip().lower() for f in FIELDS}
            if all(out[f] in FIELDS[f] for f in FIELDS):
                return out, "json"
        except json.JSONDecodeError:
            pass
    low = text.lower().replace("feature request", "feature_request")
    out = {}
    for f, allowed in FIELDS.items():
        hits = {v for v in allowed if re.search(rf"\b{re.escape(v)}\b", low)}
        if len(hits) != 1:
            return None, "unusable"
        out[f] = hits.pop()
    return out, "words"

def run_one(variant: str, split: str, ticket: dict, run: int):
    system, user = build(VARIANTS[variant], ticket["text"])
    kw = {"system": system} if system else {}
    t0 = time.perf_counter()
    r = client.messages.create(model=MODEL, max_tokens=600, thinking={"type": "disabled"},
                               messages=[{"role": "user", "content": user}], **kw)
    text = "".join(b.text for b in r.content if b.type == "text")
    got, how = parse(text)
    row = {"variant": variant, "split": split, "id": ticket["id"], "run": run, "parsed": how,
           "input": r.usage.input_tokens, "output": r.usage.output_tokens, "seconds": time.perf_counter() - t0,
           "cost": (r.usage.input_tokens * PRICE["input"] + r.usage.output_tokens * PRICE["output"]) / 1e6, "text": text}
    for f in FIELDS:
        row[f] = bool(got) and got[f] == ticket["label"][f]
    return row

print(parse('```json\n{"category": "billing", "priority": "high", "sentiment": "neutral"}\n```'))
print(parse("Category: Billing. Priority: High, because they were charged twice. Sentiment: neutral."))
print(parse("This looks like billing, high or medium priority."))

({'category': 'billing', 'priority': 'high', 'sentiment': 'neutral'}, 'json')
({'category': 'billing', 'priority': 'high', 'sentiment': 'neutral'}, 'words')
(None, 'unusable')


## 3. Run them

6 prompts × 32 tickets × 3 runs = 576 calls, 8 at a time.

In [5]:
jobs = [(v, s, t, run) for v in VARIANTS for s in ("dev", "test") for t in DATA[s] for run in (1, 2, 3)]
with ThreadPoolExecutor(8) as ex:
    rows = list(ex.map(lambda j: run_one(*j), jobs))
runs = pd.DataFrame(rows)
print(f"{len(runs)} calls | total cost ${runs['cost'].sum():.4f}")

def summary(df):
    n = len(df)
    return pd.Series({
        "usable %": round(100 * (df["parsed"] != "unusable").mean()),
        "category %": round(100 * df["category"].mean()), "priority %": round(100 * df["priority"].mean()),
        "sentiment %": round(100 * df["sentiment"].mean()),
        "all 3 right %": round(100 * (df["category"] & df["priority"] & df["sentiment"]).mean()),
        "avg output tokens": round(df["output"].mean()), "$ per 1,000 tickets": round(df["cost"].mean() * 1000, 3),
    })

table = runs.groupby(["split", "variant"], sort=False).apply(summary).reset_index()
table

576 calls | total cost $0.0380


,split,variant,usable %,category %,priority %,sentiment %,all 3 right %,avg output tokens,"$ per 1,000 tickets"
0,dev,A: task only,25.0,18.0,22.0,23.0,13.0,178.0,0.097
1,test,A: task only,33.0,31.0,31.0,25.0,19.0,144.0,0.079
2,dev,B: + format,100.0,95.0,70.0,90.0,55.0,29.0,0.031
3,test,B: + format,100.0,75.0,75.0,81.0,42.0,29.0,0.030
4,dev,C: + context,100.0,98.0,100.0,87.0,85.0,29.0,0.067
5,test,C: + context,100.0,94.0,86.0,83.0,75.0,29.0,0.066
6,dev,D: + role (full prompt),100.0,95.0,100.0,92.0,87.0,29.0,0.071
7,test,D: + role (full prompt),100.0,100.0,92.0,83.0,75.0,29.0,0.070
8,dev,D without context,100.0,95.0,83.0,90.0,68.0,29.0,0.035
9,test,D without context,100.0,83.0,72.0,81.0,47.0,29.0,0.034


## 4. One ticket, six prompts

Ticket 1 from the dev set, first run of each prompt:

In [6]:
t1 = DATA["dev"][0]
print("TICKET:", t1["text"], "\nMY LABEL:", t1["label"], "\n")
for v in VARIANTS:
    r = runs[(runs.variant == v) & (runs.split == "dev") & (runs.id == t1["id"]) & (runs.run == 1)].iloc[0]
    print(f"--- {v} | {r['output']} output tokens | parsed: {r['parsed']}")
    print(r["text"].strip()[:400] + ("..." if len(r["text"].strip()) > 400 else ""), "\n")

TICKET: Hi, I was charged twice for order ORD-48213. Please refund the extra $49.99. Thanks, jane.doe@example.com 
MY LABEL: {'category': 'billing', 'priority': 'high', 'sentiment': 'neutral'} 

--- A: task only | 284 output tokens | parsed: unusable
**Category:** Billing / Refund (duplicate charge)

**Priority:** Medium
- A duplicate charge is a clear financial error that needs fixing, but it's a single, bounded amount ($49.99), there's no sign of account compromise, and the customer is polite. It should be handled promptly, ideally within one business day, but it isn't urgent or critical.

**Sentiment:** Neutral to mildly negative
- The tone... 

--- B: + format | 30 output tokens | parsed: json
{"category": "billing", "priority": "medium", "sentiment": "neutral"} 

--- C: + context | 29 output tokens | parsed: json
{"category": "billing", "priority": "high", "sentiment": "neutral"} 

--- D: + role (full prompt) | 29 output tokens | parsed: json
{"category": "billing", "priority": "h

## 5. What each part is worth

For each part: the change in "all 3 right" when it's added (build-up) and when it's removed from the full prompt (ablation),
on both sets. Percentage points.

In [7]:
score = table.set_index(["split", "variant"])["all 3 right %"]
usable = table.set_index(["split", "variant"])["usable %"]
rows = []
for split in ("dev", "test"):
    s = lambda v: score[(split, v)]
    rows += [
        {"split": split, "part": "format",  "added (A→B)": s("B: + format") - s("A: task only"),
         "removed from D": s("D without format") - s("D: + role (full prompt)"),
         "usable % without → with": f"{usable[(split, 'A: task only')]} → {usable[(split, 'B: + format')]}"},
        {"split": split, "part": "context", "added (B→C)": s("C: + context") - s("B: + format"),
         "removed from D": s("D without context") - s("D: + role (full prompt)"), "usable % without → with": ""},
        {"split": split, "part": "role",    "added (C→D)": s("D: + role (full prompt)") - s("C: + context"),
         "removed from D": s("C: + context") - s("D: + role (full prompt)"), "usable % without → with": ""},
    ]
worth = pd.DataFrame(rows)
worth

,split,part,added (A→B),removed from D,usable % without → with,added (B→C),added (C→D)
0,dev,format,42.0,-37.0,25.0 → 100.0,NaN,NaN
1,dev,context,NaN,-19.0,,30.0,NaN
2,dev,role,NaN,-2.0,,NaN,2.0
3,test,format,23.0,-8.0,33.0 → 100.0,NaN,NaN
4,test,context,NaN,-28.0,,33.0,NaN
5,test,role,NaN,0.0,,NaN,0.0


In [8]:
# where the full prompt still disagrees with my labels (test set, any run)
full = runs[(runs.variant == "D: + role (full prompt)")]
miss = (full.groupby(["split", "id"])[list(FIELDS)].mean() < 1).reset_index()
miss = miss[miss[list(FIELDS)].any(axis=1)]
text_of = {t["id"]: t for s in ("dev", "test") for t in DATA[s]}
pd.DataFrame([{"split": r["split"], "id": r["id"], "missed": ", ".join(f for f in FIELDS if r[f]),
               "my label": text_of[r["id"]]["label"], "ticket": text_of[r["id"]]["text"][:80]} for _, r in miss.iterrows()])

,split,id,missed,my label,ticket
0,dev,9,sentiment,"{'category': 'account', 'priority': 'low', 'sentiment': 'neutral'}",How do I change the email on my account? Thanks!
1,dev,11,sentiment,"{'category': 'feature_request', 'priority': 'medium', 'sentiment': 'positive'}",It would be great if the API supported webhooks for order updates. We'd use it r
2,dev,14,sentiment,"{'category': 'shipping', 'priority': 'medium', 'sentiment': 'negative'}",Tracking for ORD-90311 hasn't updated in 9 days. Where is it?
3,dev,20,category,"{'category': 'bug', 'priority': 'high', 'sentiment': 'negative'}",The PDF invoices show the wrong VAT number since last week's update. Our account
4,test,102,sentiment,"{'category': 'billing', 'priority': 'high', 'sentiment': 'neutral'}","I got charged twice for ORD-81234, $64.00 each time. Please fix."
5,test,107,sentiment,"{'category': 'bug', 'priority': 'medium', 'sentiment': 'negative'}",The dark theme makes the chart labels hard to read. Minor thing.
6,test,111,priority,"{'category': 'account', 'priority': 'medium', 'sentiment': 'negative'}","Can't log in on iOS, it says my session expired every time. Desktop works fine."


## 6. Findings

Every number below comes from the cells above; these are the numbers the LinkedIn post uses.

In [9]:
f = {
    "model": MODEL, "sdk": anthropic.__version__, "thinking": "disabled", "calls": len(runs), "cost": round(runs["cost"].sum(), 4),
    "table": table.to_dict("records"), "worth": worth.to_dict("records"),
    "full_prompt_misses": miss.to_dict("records"),
    "example_ticket_1": {v: runs[(runs.variant == v) & (runs.split == "dev") & (runs.id == 1) & (runs.run == 1)].iloc[0][["text", "output", "parsed"]].to_dict()
                         for v in VARIANTS},
    "runs": runs.drop(columns=["text"]).to_dict("records"),
}
Path("findings.json").write_text(json.dumps(f, indent=2, default=str))
print(json.dumps({k: f[k] for k in ("calls", "cost", "table", "worth")}, indent=1, default=str))

{
 "calls": 576,
 "cost": 0.038,
 "table": [
  {
   "split": "dev",
   "variant": "A: task only",
   "usable %": 25.0,
   "category %": 18.0,
   "priority %": 22.0,
   "sentiment %": 23.0,
   "all 3 right %": 13.0,
   "avg output tokens": 178.0,
   "$ per 1,000 tickets": 0.097
  },
  {
   "split": "test",
   "variant": "A: task only",
   "usable %": 33.0,
   "category %": 31.0,
   "priority %": 31.0,
   "sentiment %": 25.0,
   "all 3 right %": 19.0,
   "avg output tokens": 144.0,
   "$ per 1,000 tickets": 0.079
  },
  {
   "split": "dev",
   "variant": "B: + format",
   "usable %": 100.0,
   "category %": 95.0,
   "priority %": 70.0,
   "sentiment %": 90.0,
   "all 3 right %": 55.0,
   "avg output tokens": 29.0,
   "$ per 1,000 tickets": 0.031
  },
  {
   "split": "test",
   "variant": "B: + format",
   "usable %": 100.0,
   "category %": 75.0,
   "priority %": 75.0,
   "sentiment %": 81.0,
   "all 3 right %": 42.0,
   "avg output tokens": 29.0,
   "$ per 1,000 tickets": 0.03
  },
  {
